# Context Chaining Meeting Analysis POC
### Multi-Agent Style Context Engineering with LangGraph, NVIDIA Nemotron, & Databricks MLflow

This Proof of Concept (POC) implements the **Context Chaining** pattern from Chapter 1 of *From Prompts to Context: Building the Semantic Blueprint* (Context Engineering).

Instead of submitting a full, raw meeting transcript into a single unconstrained prompt, the pipeline routes context through a modular **LangGraph `StateGraph`** where each node performs a targeted analytical transformation:

1. **$g_2$ (`isolate_content`)**: De-noises raw transcript by removing conversational pleasantries, coffee remarks, and greetings.
2. **$g_3$ (`new_developments`)**: Performs delta extraction comparing substantive content against prior meeting memory (`previous_summary`).
3. **$g_4$ (`implicit_dynamics`)**: Analyzes unspoken subtext, reluctance, workload pressure, and team sentiment.
4. **$g_5$ (`novel_solution`)**: Synthesizes cross-functional mitigation (combining frontend bandwidth with backend delay).
5. **$g_6$ (`summary_table`)**: Formats new developments into a clean 3-column Markdown table (`Topic | Decision/Outcome | Owner`).
6. **$g_7$ (`followup_email`)**: Produces an actionable, professional follow-up email to the team (Sarah, Tom, Maria).

**MLOps & Observability**: Every node execution, latency span, and token payload is traced automatically to **Databricks MLflow** via `mlflow.langchain.autolog()` using credentials loaded directly from your environment.

## Step 1: Environment Configuration & Databricks MLflow Tracing
Load environment credentials (`NVIDIA_API_KEY`, `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, `MLFLOW_TRACKING_URI`, `MLFLOW_EXPERIMENT_ID`) and enable autologging.

In [3]:
import os
import time
import json
import operator
import mlflow
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from dotenv import load_dotenv

# Load .env fallback while respecting existing bash environment variables
load_dotenv()

print("=== Environment & MLOps Configuration ===")
print(f"NVIDIA_API_KEY set: {bool(os.getenv('NVIDIA_API_KEY'))}")
print(f"DATABRICKS_HOST: {os.getenv('DATABRICKS_HOST') or 'Not configured'}")
print(f"MLFLOW_TRACKING_URI: {os.getenv('MLFLOW_TRACKING_URI') or 'Local mlruns'}")
print(f"MLFLOW_REGISTRY_URI: {os.getenv('MLFLOW_REGISTRY_URI') or 'Not configured'}")
print(f"MLFLOW_EXPERIMENT_ID: 4257250531416564")

# Configure Databricks MLflow Tracking
tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "databricks")
registry_uri = os.getenv("MLFLOW_REGISTRY_URI", "databricks-uc")
exp_id = "4257250531416564"

if os.getenv("DATABRICKS_HOST") and os.getenv("DATABRICKS_TOKEN"):
    mlflow.set_tracking_uri(tracking_uri)
    if registry_uri:
        try:
            mlflow.set_registry_uri(registry_uri)
        except Exception as e:
            print(f"[Notice] Registry URI setting: {e}")

if exp_id:
    mlflow.set_experiment(experiment_id=exp_id)
else:
    try:
        mlflow.set_experiment("/Shared/cot-meeting-analysis")
    except Exception:
        mlflow.set_experiment("cot-meeting-analysis")

# Auto-trace all LangChain and LangGraph operations
mlflow.langchain.autolog()
print("\n✓ MLflow tracing enabled with Databricks tracking!")

=== Environment & MLOps Configuration ===
NVIDIA_API_KEY set: True
DATABRICKS_HOST: https://dbc-41328f01-f9fe.cloud.databricks.com
MLFLOW_TRACKING_URI: databricks
MLFLOW_REGISTRY_URI: databricks-uc
MLFLOW_EXPERIMENT_ID: 4257250531416564

✓ MLflow tracing enabled with Databricks tracking!


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## Step 2: NVIDIA Chat Model Initialization (`nvidia/nemotron-3-super-120b-a12b`)
Initialize the chat client targeting NVIDIA Build NIM API endpoint (`https://integrate.api.nvidia.com/v1`) with `nvidia/nemotron-3-super-120b-a12b` and automated transient retry backoff.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

NVIDIA_MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"
NVIDIA_BASE_URL = os.getenv("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

llm = ChatOpenAI(
    model=NVIDIA_MODEL_NAME,
    api_key=NVIDIA_API_KEY,
    base_url=NVIDIA_BASE_URL,
    temperature=0.2,
    max_completion_tokens=2048,
    max_retries=5,
)

print(f"✓ Initialized LLM client with NVIDIA Build model: {NVIDIA_MODEL_NAME}")

# Run quick connectivity smoke test
t_smoke = time.perf_counter()
smoke_reply = llm.invoke([HumanMessage(content="Reply with the exact word 'READY'.")])
smoke_ms = (time.perf_counter() - t_smoke) * 1000.0
print(f"✓ Smoke test passed in {smoke_ms:.1f}ms")
print(f"Model Response:", smoke_reply.content.strip()[:150])

✓ Initialized LLM client with NVIDIA Build model: nvidia/nemotron-3-super-120b-a12b
✓ Smoke test passed in 7610.5ms
Model Response: READY


## Step 3: Input Benchmark Fixtures
Load the benchmark meeting transcript and prior baseline summary from `sample_data/`.

In [5]:
sample_transcript = """Tom: Morning all. Coffee is still kicking in.
Sarah: Morning, Tom. Right, let's jump in. Project Phoenix timeline. Tom, you said the backend components are on track?
Tom: Mostly. We hit a small snag with the payment gateway integration. It's... more complex than the docs suggested. We might need another three days.
Maria: Three days? Tom, that's going to push the final testing phase right up against the launch deadline. We don't have that buffer.
Sarah: I agree with Maria. What's the alternative, Tom?
Tom: I suppose I could work over the weekend to catch up. I'd rather not, but I can see the bind we're in.
Sarah: Appreciate that, Tom. Let's tentatively agree on that. Maria, what about the front-end?
Maria: We're good. In fact, we're a bit ahead. We have some extra bandwidth.
Sarah: Excellent. Okay, one last thing. The marketing team wants to do a big social media push on launch day. Thoughts?
Tom: Seems standard.
Maria: I think that's a mistake. A big push on day one will swamp our servers if there are any initial bugs. We should do a soft launch, invite-only for the first week, and then do the big push. More controlled.
Sarah: That's a very good point, Maria. A much safer strategy. Let's go with that. Okay, great meeting. I'll send out a summary.
Tom: Sounds good. Now, more coffee."""

previous_summary = """In our last meeting, we finalized the goals for Project Phoenix and assigned backend work to Tom and front-end to Maria."""

print("=== Input Fixtures ===")
print(f"Raw Transcript Characters: {len(sample_transcript)}")
print(f"Previous Baseline Characters: {len(previous_summary)}")
print("\n--- Raw Transcript Snippet ---")
print(sample_transcript[:250] + "...")

=== Input Fixtures ===
Raw Transcript Characters: 1303
Previous Baseline Characters: 120

--- Raw Transcript Snippet ---
Tom: Morning all. Coffee is still kicking in.
Sarah: Morning, Tom. Right, let's jump in. Project Phoenix timeline. Tom, you said the backend components are on track?
Tom: Mostly. We hit a small snag with the payment gateway integration. It's... more ...


## Step 4: Prompt Contracts ($g_2 \rightarrow g_7$)
Define the explicit modular prompt contracts adapted from Chapter 1.

In [6]:
PROMPT_G2_ISOLATE_CONTENT = """Analyze the following meeting transcript. Your task is to isolate the substantive content from the conversational noise.
- Substantive content includes: decisions made, project updates, problems raised, and strategic suggestions.
- Noise includes: greetings, pleasantries, and off-topic remarks (like coffee).
Return ONLY the substantive content.

Transcript:
---
{meeting_transcript}
---"""

PROMPT_G3_NEW_DEVELOPMENTS = """Context: The summary of our last meeting was: "{previous_summary}"

Task: Analyze the following substantive content from our new meeting. Identify and summarize ONLY the new developments, problems, or decisions that have occurred since the last meeting.

New Meeting Content:
---
{substantive_content}
---"""

PROMPT_G4_IMPLICIT_DYNAMICS = """Task: Analyze the following meeting content for implicit social dynamics and unstated feelings. Go beyond the literal words.
- Did anyone seem hesitant or reluctant despite agreeing to something?
- Were there any underlying disagreements or tensions?
- What was the overall mood?

Meeting Content:
---
{substantive_content}
---"""

PROMPT_G5_NOVEL_SOLUTION = """Context: In the meeting, Maria suggested a 'soft launch' to avoid server strain, and also mentioned her team has 'extra bandwidth'. Tom is facing a 3-day delay on the backend.

Task: Propose a novel, actionable idea that uses Maria's team's extra bandwidth to help mitigate Tom's 3-day delay. Combine these two separate pieces of information into a single solution.

Meeting Content:
---
{substantive_content}
---"""

PROMPT_G6_SUMMARY_TABLE = """Task: Create a final, concise summary of the meeting in a markdown table. Use the following information to construct the table.

- New Developments:
{new_developments}

The table should have three columns: "Topic", "Decision/Outcome", and "Owner"."""

PROMPT_G7_FOLLOWUP_EMAIL = """Task: Based on the following summary table, draft a polite and professional follow-up email to the team (Sarah, Tom, Maria).
The email should clearly state the decisions made and the action items for each person.

Summary Table:
---
{final_summary_table}
---"""

print("✓ Prompts g2, g3, g4, g5, g6, g7 defined successfully.")

✓ Prompts g2, g3, g4, g5, g6, g7 defined successfully.


## Step 5: LangGraph State Definition & Node Implementation
Define `MeetingAnalysisState` (TypedDict) with `Annotated[List[Dict[str, Any]], operator.add]` for concurrent branch reduction.

In [7]:
from langgraph.graph import StateGraph, START, END

class MeetingAnalysisState(TypedDict):
    meeting_transcript: str
    previous_summary: str
    substantive_content: Optional[str]
    new_developments: Optional[str]
    implicit_threads: Optional[str]
    novel_solution: Optional[str]
    final_summary_table: Optional[str]
    follow_up_email: Optional[str]
    step_metrics: Annotated[List[Dict[str, Any]], operator.add]

def node_isolate_content(state: MeetingAnalysisState) -> dict:
    t0 = time.perf_counter()
    prompt = PROMPT_G2_ISOLATE_CONTENT.format(meeting_transcript=state["meeting_transcript"])
    res = llm.invoke([HumanMessage(content=prompt)])
    dt = (time.perf_counter() - t0) * 1000.0
    return {
        "substantive_content": res.content.strip(),
        "step_metrics": [{"node": "g2_isolate_content", "latency_ms": round(dt, 2)}]
    }

def node_new_developments(state: MeetingAnalysisState) -> dict:
    t0 = time.perf_counter()
    prompt = PROMPT_G3_NEW_DEVELOPMENTS.format(
        previous_summary=state.get("previous_summary", ""),
        substantive_content=state["substantive_content"]
    )
    res = llm.invoke([HumanMessage(content=prompt)])
    dt = (time.perf_counter() - t0) * 1000.0
    return {
        "new_developments": res.content.strip(),
        "step_metrics": [{"node": "g3_new_developments", "latency_ms": round(dt, 2)}]
    }

def node_implicit_dynamics(state: MeetingAnalysisState) -> dict:
    t0 = time.perf_counter()
    prompt = PROMPT_G4_IMPLICIT_DYNAMICS.format(substantive_content=state["substantive_content"])
    res = llm.invoke([HumanMessage(content=prompt)])
    dt = (time.perf_counter() - t0) * 1000.0
    return {
        "implicit_threads": res.content.strip(),
        "step_metrics": [{"node": "g4_implicit_dynamics", "latency_ms": round(dt, 2)}]
    }

def node_novel_solution(state: MeetingAnalysisState) -> dict:
    t0 = time.perf_counter()
    prompt = PROMPT_G5_NOVEL_SOLUTION.format(substantive_content=state["substantive_content"])
    res = llm.invoke([HumanMessage(content=prompt)])
    dt = (time.perf_counter() - t0) * 1000.0
    return {
        "novel_solution": res.content.strip(),
        "step_metrics": [{"node": "g5_novel_solution", "latency_ms": round(dt, 2)}]
    }

def node_create_summary_table(state: MeetingAnalysisState) -> dict:
    t0 = time.perf_counter()
    prompt = PROMPT_G6_SUMMARY_TABLE.format(new_developments=state["new_developments"])
    res = llm.invoke([HumanMessage(content=prompt)])
    dt = (time.perf_counter() - t0) * 1000.0
    return {
        "final_summary_table": res.content.strip(),
        "step_metrics": [{"node": "g6_summary_table", "latency_ms": round(dt, 2)}]
    }

def node_draft_followup_email(state: MeetingAnalysisState) -> dict:
    t0 = time.perf_counter()
    prompt = PROMPT_G7_FOLLOWUP_EMAIL.format(final_summary_table=state["final_summary_table"])
    res = llm.invoke([HumanMessage(content=prompt)])
    dt = (time.perf_counter() - t0) * 1000.0
    return {
        "follow_up_email": res.content.strip(),
        "step_metrics": [{"node": "g7_follow_up_email", "latency_ms": round(dt, 2)}]
    }

print("✓ Node functions defined with Annotated list reducer for step_metrics.")

✓ Node functions defined with Annotated list reducer for step_metrics.


## Step 6: Assemble & Compile the LangGraph `StateGraph`
Construct the workflow topology connecting nodes from `START` to `END`.

In [8]:
builder = StateGraph(MeetingAnalysisState)

# Add nodes
builder.add_node("isolate_content", node_isolate_content)
builder.add_node("new_developments", node_new_developments)
builder.add_node("implicit_dynamics", node_implicit_dynamics)
builder.add_node("novel_solution", node_novel_solution)
builder.add_node("summary_table", node_create_summary_table)
builder.add_node("followup_email", node_draft_followup_email)

# Wire workflow graph
builder.add_edge(START, "isolate_content")
builder.add_edge("isolate_content", "new_developments")
builder.add_edge("isolate_content", "implicit_dynamics")
builder.add_edge("isolate_content", "novel_solution")
builder.add_edge("new_developments", "summary_table")
builder.add_edge("summary_table", "followup_email")
builder.add_edge("followup_email", END)
builder.add_edge("implicit_dynamics", END)
builder.add_edge("novel_solution", END)

graph = builder.compile()
print("✓ LangGraph StateGraph compiled successfully!")

try:
    print("\n--- Graph Mermaid Topology ---")
    print(graph.get_graph().draw_mermaid())
except Exception:
    pass

✓ LangGraph StateGraph compiled successfully!

--- Graph Mermaid Topology ---
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	isolate_content(isolate_content)
	new_developments(new_developments)
	implicit_dynamics(implicit_dynamics)
	novel_solution(novel_solution)
	summary_table(summary_table)
	followup_email(followup_email)
	__end__([<p>__end__</p>]):::last
	__start__ --> isolate_content;
	isolate_content --> implicit_dynamics;
	isolate_content --> new_developments;
	isolate_content --> novel_solution;
	new_developments --> summary_table;
	summary_table --> followup_email;
	followup_email --> __end__;
	implicit_dynamics --> __end__;
	novel_solution --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 7: End-to-End Pipeline Execution in MLflow Run
Execute the full meeting analysis pipeline inside an active MLflow run, recording duration and latency metrics per node.

In [9]:
print("=== Starting End-to-End Execution ===")

with mlflow.start_run(run_name="poc_cot_meeting_analysis_nemotron") as run:
    run_id = run.info.run_id
    print(f"Active Databricks MLflow Run ID: {run_id}")
    
    initial_state: MeetingAnalysisState = {
        "meeting_transcript": sample_transcript,
        "previous_summary": previous_summary,
        "substantive_content": None,
        "new_developments": None,
        "implicit_threads": None,
        "novel_solution": None,
        "final_summary_table": None,
        "follow_up_email": None,
        "step_metrics": [],
    }

    t_start = time.perf_counter()
    final_state = graph.invoke(initial_state)
    total_latency_ms = (time.perf_counter() - t_start) * 1000.0

    # Log summary metrics to MLflow
    mlflow.log_metric("total_pipeline_latency_ms", total_latency_ms)
    mlflow.log_param("model_name", NVIDIA_MODEL_NAME)
    for m in final_state.get("step_metrics", []):
        mlflow.log_metric(f"{m['node']}_latency_ms", m["latency_ms"])
        print(f"  ✓ Step completed: {m['node']} ({m['latency_ms']:.1f}ms)")

    print(f"\n✓ Pipeline completed successfully in {total_latency_ms:.1f}ms!")

=== Starting End-to-End Execution ===
Active Databricks MLflow Run ID: a703eecc46094a70b751c2ef31591506
  ✓ Step completed: g2_isolate_content (9011.0ms)
  ✓ Step completed: g4_implicit_dynamics (30325.5ms)
  ✓ Step completed: g3_new_developments (14399.4ms)
  ✓ Step completed: g5_novel_solution (17784.3ms)
  ✓ Step completed: g6_summary_table (3048.2ms)
  ✓ Step completed: g7_follow_up_email (4609.0ms)

✓ Pipeline completed successfully in 47007.8ms!
🏃 View run poc_cot_meeting_analysis_nemotron at: https://dbc-41328f01-f9fe.cloud.databricks.com/ml/experiments/4257250531416564/runs/a703eecc46094a70b751c2ef31591506
🧪 View experiment at: https://dbc-41328f01-f9fe.cloud.databricks.com/ml/experiments/4257250531416564


## Step 8: Generated Artifact Inspection & Quality Assertions
Inspect the de-noised content, delta state, implicit sentiment, novel solution, Markdown summary table, and follow-up email.

In [10]:
from IPython.display import display, Markdown

print("=" * 65)
print("  1. SUBSTANTIVE CONTENT (g2: Isolate Key Content)")
print("=" * 65)
print(final_state["substantive_content"])

print("\n" + "=" * 65)
print("  2. NEW DEVELOPMENTS (g3: Delta Against Baseline)")
print("=" * 65)
print(final_state["new_developments"])

print("\n" + "=" * 65)
print("  3. IMPLICIT DYNAMICS & SUBTEXT (g4)")
print("=" * 65)
print(final_state["implicit_threads"])

print("\n" + "=" * 65)
print("  4. NOVEL MITIGATION STRATEGY (g5)")
print("=" * 65)
print(final_state["novel_solution"])

print("\n" + "=" * 65)
print("  5. FINAL SUMMARY TABLE (g6)")
print("=" * 65)
display(Markdown(final_state["final_summary_table"]))

print("\n" + "=" * 65)
print("  6. ACTIONABLE FOLLOW-UP EMAIL (g7)")
print("=" * 65)
display(Markdown(final_state["follow_up_email"]))

# Verification Assertions
print("\n=== Verification Checks ===")
assert "coffee" not in final_state["substantive_content"].lower(), "Assertion Failed: 'coffee' banter was not stripped"
print("✓ De-noising check passed (chit-chat stripped)")

assert any(w in final_state["implicit_threads"].lower() for w in ["reluctan", "hesitan", "weekend", "pressure", "tension"]), "Assertion Failed: implicit dynamics not captured"
print("✓ Implicit dynamics check passed (subtext captured)")

assert "topic" in final_state["final_summary_table"].lower(), "Assertion Failed: Topic column missing"
assert "owner" in final_state["final_summary_table"].lower(), "Assertion Failed: Owner column missing"
print("✓ Summary table Markdown structure check passed")

assert all(p in final_state["follow_up_email"] for p in ["Tom", "Maria", "Sarah"]), "Assertion Failed: Missing team member in email"
print("✓ Follow-up email address check passed (Sarah, Tom, Maria included)")

print("\n🎉 ALL CONTEXT CHAINING VALIDATION CHECKS PASSED!")

  1. SUBSTANTIVE CONTENT (g2: Isolate Key Content)
Tom: Mostly. We hit a small snag with the payment gateway integration. It's... more complex than the docs suggested. We might need another three days.
Maria: Three days? Tom, that's going to push the final testing phase right up against the launch deadline. We don't have that buffer.
Tom: I suppose I could work over the weekend to catch up. I'd rather not, but I can see the bind we're in.
Sarah: Appreciate that, Tom. Let's tentatively agree on that.
Maria: We're good. In fact, we're a bit ahead. We have some extra bandwidth.
Maria: I think that's a mistake. A big push on day one will swamp our servers if there are any initial bugs. We should do a soft launch, invite-only for the first week, and then do the big push. More controlled.
Sarah: That's a very good point, Maria. A much safer strategy. Let's go with that.

  2. NEW DEVELOPMENTS (g3: Delta Against Baseline)
- **Problem:** Tom reports a snag with the payment‑gateway integration 

| Topic | Decision/Outcome | Owner |
|---|---|---|
| Payment‑gateway integration snag | Tom to work over the weekend to recover ~3 days of work (tentative) | Tom |
| Extra bandwidth/availability | Team has additional capacity; can absorb extra work | Maria |
| Launch strategy risk (big‑push) | Adopt invite‑only soft launch for the first week, then full public push | Team |


  6. ACTIONABLE FOLLOW-UP EMAIL (g7)


Subject: Follow‑up on Decisions & Action Items from Today’s Meeting  

Hi Sarah, Tom, and Maria,

Thank you all for the productive discussion earlier today. Below is a concise recap of the decisions we reached and the next steps for each of you.

| Topic | Decision / Outcome | Action Item | Owner |
|-------|-------------------|-------------|-------|
| **Payment‑gateway integration snag** | Tom will work over the weekend to recover the ~3 days of work that fell behind (tentative). | • Dedicate focused time this weekend to resolve the integration issues.<br>• Update the team on progress by end‑day Sunday. | **Tom** |
| **Extra bandwidth/availability** | The team has additional capacity and can absorb extra work as needed. | • Keep an eye on workload distribution and flag any bottlenecks early.<br>• Be ready to lend support to the payment‑gateway effort if required. | **Maria** |
| **Launch‑strategy risk (big‑push)** | Adopt an invite‑only soft launch for the first week, followed by a full public push. | • Finalise the invite‑list and communication plan for the soft launch.<br>• Prepare marketing assets and monitoring dashboards for the subsequent public rollout.<br>• Coordinate with Tom and Maria to ensure the gateway is stable before the soft launch goes live. | **Sarah** (lead) – with support from Tom and Maria as outlined above |

Please let me know if anything is unclear or if you anticipate any obstacles that might affect these timelines. I’ll send a brief status‑check reminder on Friday to keep everyone aligned.

Thanks again for your collaboration.

Best regards,  
[Your Name]  
[Your Title]  
[Your Contact Information]


=== Verification Checks ===
✓ De-noising check passed (chit-chat stripped)
✓ Implicit dynamics check passed (subtext captured)
✓ Summary table Markdown structure check passed
✓ Follow-up email address check passed (Sarah, Tom, Maria included)

🎉 ALL CONTEXT CHAINING VALIDATION CHECKS PASSED!
